# Solution 05 — streaming + anomalies

In [ ]:
from confluent_kafka import Producer
from datetime import datetime
import json, time, random

producer = Producer({'bootstrap.servers': 'redpanda:29092', 'client.id': 'streaming-producer'})
houses  = ['haus_a', 'haus_b', 'haus_c', 'haus_d']
sensors = {
    'strom':  {'unit': 'kWh',   'min': 0.5, 'max': 15.0},
    'wasser': {'unit': 'Liter', 'min': 0.1, 'max': 50.0},
}

## Step 1 — continuous stream

Notice we `flush()` after every message here so the consumer sees events one-by-one instead of in batches. That's nice for the demo but not what you'd do in production — let librdkafka batch.

In [ ]:
count = 0
print(f'{"Time":>10} | {"#":>4} | {"Topic":>6} | {"Key":>7} | {"Value":>8}')
print('-' * 50)

try:
    while True:
        house = random.choice(houses)
        topic = random.choice(list(sensors.keys()))
        cfg   = sensors[topic]
        value = round(random.uniform(cfg['min'], cfg['max']), 2)
        event = json.dumps({
            'sensor':    topic,
            'haus':      house,
            'wert':      value,
            'einheit':   cfg['unit'],
            'timestamp': time.time(),
        })
        producer.produce(topic, key=house.encode(), value=event.encode())
        producer.flush()
        count += 1
        ts = datetime.now().strftime('%H:%M:%S')
        print(f'{ts:>10} | {count:>4} | {topic:>6} | {house:>7} | {value:>8.2f}')
        time.sleep(random.uniform(0.5, 2.0))
except KeyboardInterrupt:
    producer.flush()
    print(f'\nStopped. Total: {count}')

## Task A — burst (no sleep)

Compare the throughput line in your consumer: a 50-event burst completes here in well under a second. That's Kafka's whole marketing line in action.

In [ ]:
for _ in range(50):
    house = random.choice(houses)
    topic = random.choice(list(sensors.keys()))
    cfg   = sensors[topic]
    value = round(random.uniform(cfg['min'], cfg['max']), 2)
    event = json.dumps({'sensor': topic, 'haus': house, 'wert': value,
                        'einheit': cfg['unit'], 'timestamp': time.time()})
    producer.produce(topic, key=house.encode(), value=event.encode())
producer.flush()   # one flush, AFTER the loop
print('Burst sent.')

## Task B — anomaly simulation

We tag the message itself rather than recomputing the threshold downstream. The consumer in exercise 06 will use that flag.

In [ ]:
n_anoms = 0
for i in range(30):
    house  = random.choice(houses)
    topic  = random.choice(list(sensors.keys()))
    cfg    = sensors[topic]
    is_anom = random.random() < 0.2
    if is_anom:
        # 3-8x the normal max — clearly bogus
        value = round(random.uniform(cfg['max'] * 3, cfg['max'] * 8), 2)
        n_anoms += 1
    else:
        value = round(random.uniform(cfg['min'], cfg['max']), 2)
    event = json.dumps({'sensor': topic, 'haus': house, 'wert': value,
                        'einheit': cfg['unit'], 'timestamp': time.time(),
                        'anomaly': is_anom})
    producer.produce(topic, key=house.encode(), value=event.encode())
    producer.flush()
    note = '<-- ANOMALY' if is_anom else ''
    print(f'{i+1:>3} | {topic:>6} | {house:>7} | {value:>8.2f} | {note}')
print(f'\n{n_anoms}/30 events were anomalies.')